# MovieLens 10M Recommender System — Final Project Planning Document

**Author:** Nana Y. Frimpong  
**Course:** Data Science — Final Project Proposal

---

## 1. Introduction

The goal of this project is to build a recommender system that produces quality movie recommendations by extracting insight from a large, sparse ratings dataset. Rather than a single-technique approach, this proposal outlines a **hybrid recommender**: a collaborative filtering model built via matrix factorization (SVD), complemented by a content-based model built from user-generated tags and genre metadata. This combination allows the system to recommend based on *what similar users liked* (collaborative) as well as *what a movie is actually about* (content), and sets up a natural cold-start mitigation strategy for movies with few ratings.


## 2. Dataset Description

This project uses the **MovieLens 10M** dataset, published by GroupLens Research (University of Minnesota), a standard benchmark dataset for recommender systems research.

**Source:** https://grouplens.org/datasets/movielens/10m/

| File | Description |
|---|---|
| `ratings.dat` | 10,000,054 ratings from 69,878 users on 10,677 movies (0.5–5.0 stars, half-star increments) |
| `movies.dat` | Movie titles and pipe-separated genres |
| `tags.dat` | 95,580 free-text tags applied by users to movies |

This satisfies the assignment's size requirement on both fronts: **10M+ ratings** and **69,878 users / 10,677 items**, both comfortably above the 10k threshold.


## 3. Unique Element

Beyond the standard user–item ratings matrix, this project incorporates the **`tags.dat`** file — free-text tags users applied to movies (e.g. "twist ending", "based on a book", "dark comedy"). These are combined with genre metadata into a per-movie text profile and vectorized with **TF-IDF**, giving each movie a content-based feature representation independent of the ratings matrix.

This supports:
- A **content-based similarity model** ("movies similar to X") that works even for movies with very few ratings (cold-start).
- A future **hybrid blending step** that combines collaborative filtering scores with content similarity, which will be developed further for the final deliverable.


## 4. Proposed Methodology

1. **Baseline model** — global mean and user/item mean-based prediction, to establish a naive benchmark.
2. **Collaborative filtering** — matrix factorization via truncated SVD on the user–item sparse ratings matrix (mean-centered per user), predicting unseen ratings via reconstructed low-rank factors. This is the "advanced mathematical technique" component in place of Spark/distributed computing, per the assignment's alternative option.
3. **Content-based filtering** — TF-IDF vectorization of tag + genre text per movie, with cosine similarity used to surface similar titles.
4. **(Planned for final deliverable) Hybrid blending** — weighted combination of CF and content scores, tuned to improve cold-start and long-tail recommendation quality.

## 5. Evaluation Plan

- **Quantitative:** train/test split (80/20) of ratings; RMSE and MAE on held-out ratings for the CF model, compared against the baseline.
- **Qualitative:** nearest-neighbor inspection for the content model (e.g., "movies similar to Toy Story") to sanity-check that recommendations are sensible.
- **Final deliverable will add:** precision@k / recall@k for top-N recommendation quality, not just rating prediction accuracy.


## 6. Feasibility Check

A quick load of the raw data files, confirming the dataset is accessible, clean, and meets the stated size requirements before committing to the full build.

In [1]:
import pandas as pd, os

data_path = r'/Users/nanafrimpong/Desktop/SPS Summer 2026/yyyooo/files'

# Row counts (fast, low-memory) to confirm dataset size before loading
with open(os.path.join(data_path, "ratings.dat")) as f:
    n_ratings = sum(1 for _ in f)

movies = pd.read_csv(os.path.join(data_path, "movies.dat"), sep='::', engine='python',
                      names=['movieId','title','genres'], encoding='latin-1')
tags = pd.read_csv(os.path.join(data_path, "tags.dat"), sep='::', engine='python',
                    names=['userId','movieId','tag','timestamp'], encoding='latin-1')

# Sample of ratings (full file is 10M rows / ~250MB; sampled here for a quick feasibility check)
ratings_sample = pd.read_csv(os.path.join(data_path, "ratings.dat"), sep='::', engine='python', nrows=100000,
                              names=['userId','movieId','rating','timestamp'])

print(f"Total ratings rows (full file): {n_ratings:,}")
print(f"Movies: {len(movies):,} rows")
print(f"Tags:   {len(tags):,} rows")
print(f"\nSample of ratings.dat:")
ratings_sample.head()

Total ratings rows (full file): 10,000,054
Movies: 10,681 rows
Tags:   95,580 rows

Sample of ratings.dat:


,userId,movieId,rating,timestamp
0,1,122,5.0,838985046
1,1,185,5.0,838983525
2,1,231,5.0,838983392
3,1,292,5.0,838983421
4,1,316,5.0,838983392


## 7. Next Steps / Timeline

- Complete full exploratory data analysis (ratings distribution, user/item activity, genre and temporal patterns)
- Implement and evaluate the SVD-based collaborative filtering model
- Implement and evaluate the TF-IDF content-based model
- Build the hybrid blending logic and top-N recommendation evaluation (precision@k / recall@k)
- Package the full build into the final project notebook for submission
